# B2 - Change Streams with PyMongo (8 min)
**Slides 35-38.** We open a change stream on `shop.orders`, make changes, read the events, then stop and **resume** from a saved token.

Change streams need a replica set (slide 35) - our `mongo` container runs a 1-member replica set `rs0`.

In [ ]:
import os
from pymongo import MongoClient

# Inside dev_env the host is "mongo". Jupyter in this image starts through sudo,
# which drops the container's MONGO_URI variable, so we fall back to the same value.
URI = os.getenv("MONGO_URI", "mongodb://mongo:27017/?directConnection=true")
client = MongoClient(URI, serverSelectionTimeoutMS=5000)
coll = client["shop"]["orders"]
print("connected to", URI, "| orders:", coll.count_documents({}))

## 1. Open the stream (slide 35, 36)
`full_document="updateLookup"` also returns the current whole document on updates (slide 36).

In [ ]:
pipeline = [{"$match": {"operationType": {"$in": ["insert", "update", "replace", "delete"]}}}]
stream = coll.watch(pipeline, full_document="updateLookup")
print("stream open")

## 2. Make changes (insert + update)
In class you can do this in mongosh instead; here we do it from Python so the notebook runs on its own.

In [ ]:
coll.insert_one({"order_id": "NB-1", "status": "created"})
coll.update_one({"order_id": "NB-1"}, {"$set": {"status": "paid"}})
print("done")

## 3. Read the events (with a timeout)
A change stream is a cursor that never ends. `try_next()` returns `None` when nothing new arrived yet,
so we loop for a few seconds instead of blocking forever.

In [ ]:
import time
from bson import json_util

def read_events(stream, seconds=5):
    deadline = time.time() + seconds
    while time.time() < deadline:
        event = stream.try_next()
        if event is None:
            time.sleep(0.2)
            continue
        print(event["operationType"].upper(), event["documentKey"]["_id"])
        if event["operationType"] == "update":
            print("  updatedFields:", json_util.dumps(event["updateDescription"]["updatedFields"]))
        print("  fullDocument :", json_util.dumps(event.get("fullDocument")))

read_events(stream)
token = stream.resume_token        # the bookmark (slide 37)
stream.close()
print("stream closed, token saved:", token is not None)

Expected (ids will differ):
```
INSERT <ObjectId>
  fullDocument : {"_id": {"$oid": "..."}, "order_id": "NB-1", "status": "created"}
UPDATE <ObjectId>
  updatedFields: {"status": "paid"}
  fullDocument : {"_id": {"$oid": "..."}, "order_id": "NB-1", "status": "paid"}
stream closed, token saved: True
```

**Reflection (slide 36):** the update event alone only says `status: paid`. What else does your Silver table need to already have?

## 4. Delete while nobody is listening, then resume (slide 37)
The stream is closed - this is the "consumer crashed" moment.

In [ ]:
coll.delete_one({"order_id": "NB-1"})

with coll.watch(pipeline, full_document="updateLookup", resume_after=token) as stream:
    read_events(stream, seconds=3)

Expected:
```
DELETE <ObjectId>
  fullDocument : null
```
The delete happened while the stream was closed and still arrives, because we resumed after the saved token.
The delete event has only the `_id` - no document.

**Reflection (slide 37):** if the consumer crashes *after* printing an event but *before* saving the token, what happens on restart?

## 5. Live version in a terminal (slide 38)
`watch_orders.py` is the same loop, but it saves the token to `resume_token.json` after every event.
Open a terminal on your laptop:
```bash
docker exec -it dev_env bash
cd /home/developer/lab
python3 watch_orders.py --from-scratch
```
Then follow task B2 in the handout (insert / update in mongosh, Ctrl+C, delete, restart without `--from-scratch`).